# Step 1: Data Anatomy & The Message Passing Framework

**Goal:** Understand the QM9 dataset structure and establish the theoretical foundation for Message Passing Neural Networks (MPNNs).

---

## 1. Theoretical Background: The General MPNN Framework

Based on **"Neural Message Passing for Quantum Chemistry"** (Gilmer et al., 2017), a general Message Passing Neural Network operates on a molecular graph $G = (V, E)$ where:
- $V$ = set of atoms (nodes)
- $E$ = set of bonds (edges)
- Each node $v$ has a feature vector $h_v^0$ (e.g., atom type, charge)
- Each edge $(v, w)$ has a feature vector $e_{vw}$ (e.g., bond distance, bond type)

### 1.1 Message Passing Phase

The network updates node representations iteratively for $T$ timesteps:

#### **Message Function** (Aggregation)
At each timestep $t$, node $v$ receives messages from its neighbors:

$$
m_v^{t+1} = \sum_{w \in \mathcal{N}(v)} M_t\left(h_v^t, h_w^t, e_{vw}\right)
$$

**Notation Mapping:**
- $h_v^t$ → `node_feats[v]` (feature vector for node $v$ at timestep $t$)
- $h_w^t$ → `node_feats[w]` (feature vector for neighbor $w$)
- $e_{vw}$ → `edge_feats[e]` (edge feature, e.g., bond distance)
- $m_v^{t+1}$ → `messages[v]` (aggregated message for node $v$)
- $M_t$ → A neural network (e.g., MLP) that computes edge-conditioned messages

#### **Update Function** (Node State Update)
The node's hidden state is updated using the aggregated message:

$$
h_v^{t+1} = U_t\left(h_v^t, m_v^{t+1}\right)
$$

**Notation Mapping:**
- $U_t$ → A recurrent unit (GRU) or MLP
- $h_v^{t+1}$ → Updated `node_feats[v]`

### 1.2 Readout Phase

After $T$ message passing steps, the final node representations $\{h_v^T | v \in G\}$ are aggregated into a graph-level representation:

$$
\hat{y} = R\left(\{h_v^T | v \in G\}\right)
$$

**Notation Mapping:**
- $R$ → Readout function (e.g., sum, mean, or Set2Set)
- $\hat{y}$ → `prediction` (scalar value, e.g., HOMO-LUMO gap)

Common choices:
- Sum: $R = \sum_{v \in G} \sigma\left(g(h_v^T)\right)$
- Set2Set: An attention-based mechanism (used in Gilmer et al.)

---

## 2. Dataset: QM9 (Quantum Machines 9)

QM9 contains **~134k organic molecules** with quantum chemistry properties computed via DFT.

**Target Properties (12 tasks):**
1. $\mu$ - Dipole moment
2. $\alpha$ - Isotropic polarizability
3. $\epsilon_{\text{HOMO}}$ - Highest Occupied Molecular Orbital energy
4. $\epsilon_{\text{LUMO}}$ - Lowest Unoccupied Molecular Orbital energy
5. $\Delta \epsilon = \epsilon_{\text{LUMO}} - \epsilon_{\text{HOMO}}$ - **HOMO-LUMO gap** (our primary target)
6. $\langle R^2 \rangle$ - Electronic spatial extent
7. $\text{ZPVE}$ - Zero point vibrational energy
8. $U_0$ - Internal energy at 0K
9-12. $U, H, G, c_v$ - Thermodynamic properties

---

In [ ]:
# Install dependencies (run once)
# !pip install torch torch-geometric rdkit matplotlib

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from torch_geometric.datasets import QM9
from torch_geometric.data import Data
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

logger.info(f"PyTorch version: {torch.__version__}")
logger.info(f"CUDA available: {torch.cuda.is_available()}")

## 3. Loading the QM9 Dataset

In [ ]:
# Download and load QM9
# This will download ~3.5GB of data to './data/QM9'
dataset = QM9(root='./data/QM9')

logger.info(f"Dataset size: {len(dataset)} molecules")
logger.info(f"Number of features per atom (node features): {dataset.num_features}")
logger.info(f"Number of targets (molecular properties): {dataset.num_classes}")

## 4. Data Anatomy: Inspecting a Single Molecule

Each molecule in QM9 is represented as a `Data` object with the following attributes:

- **`data.x`**: Node feature matrix $\mathbf{X} \in \mathbb{R}^{N \times F}$
  - $N$ = number of atoms
  - $F$ = 11 features per atom (atomic number, chirality, etc.)
  - Maps to: $h_v^0$ in the MPNN framework

- **`data.pos`**: 3D atomic coordinates $\mathbf{P} \in \mathbb{R}^{N \times 3}$
  - $(x, y, z)$ positions in Ångströms
  - Used to compute edge features $e_{vw}$ (distances)

- **`data.edge_index`**: Graph connectivity $\mathbf{E} \in \mathbb{N}^{2 \times M}$
  - $M$ = number of edges (bonds)
  - Format: `[[source_nodes], [target_nodes]]`
  - Defines $\mathcal{N}(v)$ (neighbors of node $v$)

- **`data.edge_attr`**: Edge feature matrix $\mathbf{E}_{\text{attr}} \in \mathbb{R}^{M \times E_f}$
  - $E_f$ = 4 features per edge (bond type encoded)
  - Maps to: $e_{vw}$ in the MPNN framework

- **`data.y`**: Target properties $\mathbf{y} \in \mathbb{R}^{1 \times 19}$
  - 19 quantum properties (some are derived from others)
  - Index 4 → HOMO-LUMO gap ($\Delta \epsilon$)

In [ ]:
# Select a molecule (e.g., index 42)
mol_idx = 42
data = dataset[mol_idx]

logger.info(f"\n{'='*60}")
logger.info(f"Molecule {mol_idx} - Data Structure")
logger.info(f"{'='*60}")

logger.info(f"\n[Node Features] data.x:")
logger.info(f"  Shape: {data.x.shape} → [{data.x.shape[0]} atoms, {data.x.shape[1]} features/atom]")
logger.info(f"  Type: {data.x.dtype}")
logger.info(f"  Sample (first atom): {data.x[0]}")

logger.info(f"\n[3D Coordinates] data.pos:")
logger.info(f"  Shape: {data.pos.shape} → [{data.pos.shape[0]} atoms, 3D coordinates]")
logger.info(f"  Sample (first atom): {data.pos[0]} Å")

logger.info(f"\n[Edge Connectivity] data.edge_index:")
logger.info(f"  Shape: {data.edge_index.shape} → [2, {data.edge_index.shape[1]} edges]")
logger.info(f"  Sample (first 5 edges):")
logger.info(f"    Source nodes: {data.edge_index[0, :5].tolist()}")
logger.info(f"    Target nodes: {data.edge_index[1, :5].tolist()}")

logger.info(f"\n[Edge Features] data.edge_attr:")
logger.info(f"  Shape: {data.edge_attr.shape} → [{data.edge_attr.shape[0]} edges, {data.edge_attr.shape[1]} features/edge]")
logger.info(f"  Sample (first edge): {data.edge_attr[0]}")

logger.info(f"\n[Target Properties] data.y:")
logger.info(f"  Shape: {data.y.shape} → [19 quantum properties]")
logger.info(f"  HOMO-LUMO gap (index 4): {data.y[0, 4]:.4f} eV")
logger.info(f"  Internal energy U_0 (index 7): {data.y[0, 7]:.4f} eV")

logger.info(f"\n{'='*60}\n")

## 5. Visualization: Molecular Structure in 3D

We'll visualize the molecule as a 3D scatter plot where:
- **Points** = atoms (colored by atomic number)
- **Lines** = bonds (from `edge_index`)

In [ ]:
def visualize_molecule_3d(data: Data, title: str = "Molecular Structure"):
    """
    Visualize a molecule in 3D using atomic coordinates and bonds.
    
    Args:
        data: PyG Data object containing pos and edge_index
        title: Plot title
    """
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Extract coordinates
    pos = data.pos.numpy()  # [N_atoms, 3]
    x, y, z = pos[:, 0], pos[:, 1], pos[:, 2]
    
    # Extract atomic numbers (first feature in data.x)
    atomic_nums = data.x[:, 0].numpy()
    
    # Color map: H=1→gray, C=6→black, N=7→blue, O=8→red, F=9→green
    color_map = {1: 'gray', 6: 'black', 7: 'blue', 8: 'red', 9: 'green'}
    colors = [color_map.get(int(z), 'purple') for z in atomic_nums]
    
    # Plot atoms
    ax.scatter(x, y, z, c=colors, s=200, alpha=0.8, edgecolors='k', linewidths=1.5)
    
    # Plot bonds
    edge_index = data.edge_index.numpy()
    for i in range(edge_index.shape[1]):
        src, dst = edge_index[0, i], edge_index[1, i]
        # Only draw each bond once (avoid duplicates in undirected graph)
        if src < dst:
            ax.plot(
                [pos[src, 0], pos[dst, 0]],
                [pos[src, 1], pos[dst, 1]],
                [pos[src, 2], pos[dst, 2]],
                'k-', alpha=0.3, linewidth=1
            )
    
    ax.set_xlabel('X (Å)', fontsize=12)
    ax.set_ylabel('Y (Å)', fontsize=12)
    ax.set_zlabel('Z (Å)', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Add legend
    legend_elements = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='gray', markersize=10, label='H'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=10, label='C'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='blue', markersize=10, label='N'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='red', markersize=10, label='O'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='green', markersize=10, label='F'),
    ]
    ax.legend(handles=legend_elements, loc='upper right')
    
    plt.tight_layout()
    plt.show()


# Visualize molecule 42
visualize_molecule_3d(
    data, 
    title=f"Molecule {mol_idx} | {data.x.shape[0]} atoms, {data.edge_index.shape[1]//2} bonds"
)

## 6. Preprocessing: From Coordinates to Edge Features

**Critical Insight:** The QM9 dataset provides `edge_attr` (bond type), but **NOT bond distances**.

In the MPNN paper, edge features $e_{vw}$ include:
1. **Euclidean distance** between atoms $v$ and $w$:
   $$d_{vw} = \|\mathbf{p}_v - \mathbf{p}_w\|_2$$
   where $\mathbf{p}_v$ is the 3D position of atom $v$.

2. **Radial Basis Function (RBF) expansion**:
   $$
   e_{vw}^{\text{RBF}} = \left[\exp\left(-\frac{(d_{vw} - \mu_k)^2}{\gamma}\right)\right]_{k=1}^K
   $$
   where $\mu_k \in [0, 6]$ Å are $K$ evenly spaced centers.

**Notation Mapping:**
- $d_{vw}$ → `edge_distances[e]` (scalar distance for edge $e$)
- $e_{vw}^{\text{RBF}}$ → `edge_rbf[e]` (vector of $K$ RBF features)

Let's implement this preprocessing step:

In [ ]:
def compute_edge_distances(data: Data) -> torch.Tensor:
    """
    Compute Euclidean distances for all edges in a molecular graph.
    
    Args:
        data: PyG Data object with attributes:
            - pos: [N_atoms, 3] - 3D coordinates
            - edge_index: [2, N_edges] - graph connectivity
    
    Returns:
        edge_distances: [N_edges] - Euclidean distance for each edge
    
    Math:
        For edge e = (v, w):
        d_e = || pos[v] - pos[w] ||_2
    """
    # Extract source and target node positions
    src_pos = data.pos[data.edge_index[0]]  # [N_edges, 3]
    dst_pos = data.pos[data.edge_index[1]]  # [N_edges, 3]
    
    # Compute L2 distance
    distances = torch.norm(src_pos - dst_pos, p=2, dim=1)  # [N_edges]
    
    return distances


def rbf_expansion(
    distances: torch.Tensor, 
    num_rbf: int = 64, 
    cutoff: float = 6.0
) -> torch.Tensor:
    """
    Expand distances using Radial Basis Functions (RBF).
    
    Args:
        distances: [N_edges] - Edge distances in Ångströms
        num_rbf: Number of RBF centers (K)
        cutoff: Maximum distance to consider (in Å)
    
    Returns:
        rbf_features: [N_edges, num_rbf] - RBF-expanded edge features
    
    Math:
        rbf_k(d) = exp(-(d - μ_k)^2 / γ)
        where μ_k = k * cutoff / num_rbf for k = 0, ..., num_rbf-1
              γ = (cutoff / num_rbf)^2
    """
    # Define RBF centers: evenly spaced from 0 to cutoff
    centers = torch.linspace(0, cutoff, num_rbf, device=distances.device)  # [num_rbf]
    
    # Gamma controls the width of the RBF
    gamma = (cutoff / num_rbf) ** 2
    
    # Compute RBF features: [N_edges, num_rbf]
    # Broadcasting: [N_edges, 1] - [1, num_rbf] → [N_edges, num_rbf]
    rbf = torch.exp(-((distances.unsqueeze(-1) - centers) ** 2) / gamma)
    
    return rbf


# Demonstrate on molecule 42
distances = compute_edge_distances(data)
rbf_features = rbf_expansion(distances, num_rbf=64, cutoff=6.0)

logger.info(f"\nEdge Distance Computation:")
logger.info(f"  Edge distances shape: {distances.shape} → [N_edges]")
logger.info(f"  Min distance: {distances.min():.3f} Å")
logger.info(f"  Max distance: {distances.max():.3f} Å")
logger.info(f"  Mean distance: {distances.mean():.3f} Å")

logger.info(f"\nRBF Expansion:")
logger.info(f"  RBF features shape: {rbf_features.shape} → [N_edges, 64]")
logger.info(f"  Sample (first edge, first 10 RBF components): {rbf_features[0, :10]}")

## 7. Summary: Data → Math → Code Mapping

| **Mathematical Symbol** | **Meaning** | **Code Variable** | **Shape** |
|-------------------------|-------------|-------------------|------------|
| $G = (V, E)$ | Molecular graph | `data` | - |
| $h_v^0$ | Initial atom features | `data.x` | `[N_atoms, 11]` |
| $\mathbf{p}_v$ | 3D coordinates | `data.pos` | `[N_atoms, 3]` |
| $\mathcal{N}(v)$ | Neighbors of $v$ | `data.edge_index` | `[2, N_edges]` |
| $e_{vw}$ | Edge features | `rbf_features` | `[N_edges, 64]` |
| $d_{vw}$ | Bond distance | `distances` | `[N_edges]` |
| $\hat{y}$ | Predicted property | TBD (Step 2+) | `[1]` |
| $y$ | Ground truth | `data.y[:, 4]` (HOMO-LUMO) | `[1]` |

---

## Next Steps

**Step 2:** Implement a **Topological GCN** (ignoring geometry) to establish a baseline.
- **Hypothesis:** "Without bond distances, the model cannot learn chemistry."
- **Expected Result:** High validation error (>1 eV for HOMO-LUMO gap).

**Key Question to Ponder:** 
Can a neural network predict quantum properties using only the molecular graph topology (atom types + bond existence), or is geometric information (bond lengths) essential?